# 00 · Configuración, salidas de Cell Ranger y el objeto AnnData
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kur1sutaru/ISMB-LATAM/blob/main/notebooks/es/00_setup_and_cellranger.ipynb)

**De Células a Clústeres: Introducción al Análisis de Single-Cell RNA-seq en Python**  
ISCB-LATAM SolBio SPBBC 2026 · Lima, Perú · 9 de noviembre de 2026  
Instructora: Dra. Cristal Villalba Silva (UT Health San Antonio) · [github.com/Kur1sutaru](https://github.com/Kur1sutaru)

⏱ **Tiempo en el tutorial:** 9:20–9:40 (20 min)

**Al final de este notebook podrás:**
- Ejecutar código en Google Colab
- Explicar qué hace Cell Ranger, desde los archivos FASTQ hasta la matriz de conteos
- Reconocer los archivos de la carpeta `outs/` de Cell Ranger y cuáles necesitas
- Cargar una matriz de conteos en Python y navegar un objeto `AnnData`

> 🟢 Las secciones **Core** se ven en vivo. 🔎 Las secciones **Extra** son para estudiar después del tutorial.

## Cómo usar Google Colab
- Cada caja gris es una **celda de código**. Haz clic en ella y presiona `Shift + Enter` para ejecutarla.
- Las celdas se ejecutan **en orden**. Si algo falla, usa *Entorno de ejecución → Reiniciar sesión y ejecutar todo*.
- Colab gratuito te da una máquina con CPU y ~12 GB de RAM. Es suficiente para este tutorial; no se necesita GPU.
- Las máquinas de Colab son temporales: los archivos desaparecen al terminar la sesión (salvo que uses Google Drive).

## 0. Configuración 🟢
Ejecuta primero la celda de abajo (▶ o `Shift + Enter`). En Google Colab instala los paquetes (~1 min);
en tu computadora asume que ya ejecutaste `pip install -r requirements.txt`.

**Opcional:** cambia `USE_DRIVE = True` para guardar tus resultados en Google Drive y reutilizarlos en
los siguientes notebooks. Si lo dejas en `False`, cada notebook reconstruye lo que necesita automáticamente.

In [ ]:
import os, sys, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "scanpy[leiden,scrublet]>=1.10"

USE_DRIVE = False  # True = guardar resultados en Google Drive
WORKDIR = "sc_tutorial"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/sc_tutorial"
os.makedirs(WORKDIR, exist_ok=True)

# Funciones auxiliares del tutorial (descarga + checkpoints)
if os.path.exists("../../scripts/sc_tutorial.py"):          # ejecutando desde un clon local
    sys.path.insert(0, os.path.abspath("../../scripts"))
elif not os.path.exists("sc_tutorial.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Kur1sutaru/ISMB-LATAM/main/scripts/sc_tutorial.py", "sc_tutorial.py")
import sc_tutorial as tut

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sc.settings.figdir = os.path.join(WORKDIR, "figures")
print("scanpy", sc.__version__, "| working directory:", os.path.abspath(WORKDIR))

## 1. Del secuenciador a la matriz de conteos: Cell Ranger 🟢

Un experimento de 10x Genomics te entrega **archivos FASTQ**. Cada par de lecturas contiene:
- **Read 1**: el **código de barras celular** (qué gota/célula, 16 pb) + el **UMI** (qué molécula original, 10–12 pb);
- **Read 2**: la secuencia de cDNA (qué gen).

**Cell Ranger** (`cellranger count`) convierte esas lecturas en una **matriz de conteos células × genes**:

1. **Alinea** el Read 2 al genoma/transcriptoma de referencia (usa STAR internamente).
2. **Corrige los códigos de barras** con la lista de códigos válidos de 10x.
3. **Cuenta UMIs**: lecturas con el mismo código + UMI + gen son copias de PCR de una molécula → se cuentan una vez.
4. **Identifica células**: separa las gotas que contienen una célula de las gotas vacías (solo RNA ambiental).

Cell Ranger no se ejecuta en Colab (necesita un servidor Linux con ~64 GB de RAM). Un comando típico es:

```bash
cellranger count --id=sample1 \
    --transcriptome=refdata-gex-GRCh38-2024-A \
    --fastqs=/ruta/a/fastqs --sample=sample1 \
    --create-bam=true --localcores=16 --localmem=64
```

Alternativas de código abierto que producen el mismo tipo de matriz: **STARsolo**, **alevin-fry / simpleaf**,
**kallisto | bustools**. Suelen ser más rápidas y se pueden ejecutar en cualquier lugar.

### ¿Qué hay dentro de `outs/`?

| Archivo | Qué es | ¿Lo necesitas? |
|---|---|---|
| `web_summary.html` | Reporte interactivo de control de calidad | **Ábrelo siempre primero** |
| `metrics_summary.csv` | Las mismas métricas en una tabla | Sí, para comparar muestras |
| `filtered_feature_bc_matrix/` (`barcodes.tsv.gz`, `features.tsv.gz`, `matrix.mtx.gz`) | Conteos solo de las **células identificadas** | **Sí: es lo que analizamos** |
| `filtered_feature_bc_matrix.h5` | La misma matriz en un solo archivo HDF5 | Sí (lo más fácil de cargar) |
| `raw_feature_bc_matrix/` | Conteos de **todos** los códigos, incluidas las gotas vacías | Para eliminar RNA ambiental (SoupX, CellBender) |
| `possorted_genome_bam.bam` | Lecturas alineadas | Rara vez (velocidad de RNA, genotipado) |
| `molecule_info.h5` | Información por molécula | Para agregar/submuestrear muestras |
| `cloupe.cloupe` | Archivo para la aplicación Loupe Browser | Exploración visual rápida |

**Métricas a revisar en `web_summary.html`:** número estimado de células, mediana de genes por célula,
fracción de lecturas en células (> 70 %), lecturas mapeadas con confianza al transcriptoma (> 60 %),
saturación de secuenciación y la forma del **gráfico de rango de códigos de barras ("rodilla")**.

## 2. Descargar una salida real de Cell Ranger 🟢
Usamos **PBMC 3k**: ~2.700 células mononucleares de sangre periférica de un donante sano, secuenciadas por 10x
Genomics (públicas, CC BY 4.0). Es el "hola mundo" del análisis de single-cell, así que puedes comparar tus
resultados con los tutoriales de Scanpy y Seurat. Este conjunto se generó con una versión antigua de Cell
Ranger, por eso los archivos se llaman `genes.tsv` en vez de `features.tsv.gz`, pero la estructura es la misma.

In [ ]:
import tarfile
from pathlib import Path

data_dir = Path(WORKDIR) / "data"
tar_path = tut.download(tut.URLS["pbmc3k"], data_dir / "pbmc3k_filtered_gene_bc_matrices.tar.gz")
with tarfile.open(tar_path) as fh:
    fh.extractall(data_dir)

mtx_dir = data_dir / "filtered_gene_bc_matrices" / "hg19"
for f in sorted(mtx_dir.iterdir()):
    print(f"{f.name:15s} {f.stat().st_size / 1e6:6.1f} MB")

Veamos los tres archivos. `barcodes.tsv` = una célula por línea; `genes.tsv` = ID de Ensembl + símbolo del gen;
`matrix.mtx` = una matriz **dispersa** guardada como tripletes *(gen, célula, conteo)*, porque >90 % de los valores son cero.

In [ ]:
for name in ["barcodes.tsv", "genes.tsv", "matrix.mtx"]:
    print(f"--- {name} ---")
    with open(mtx_dir / name) as fh:
        for _ in range(4):
            print(fh.readline().rstrip())

## 3. Cargar la matriz en un objeto `AnnData` 🟢
`sc.read_10x_mtx()` lee la carpeta. Para salidas recientes de Cell Ranger también puedes usar
`sc.read_10x_h5("filtered_feature_bc_matrix.h5")`.

In [ ]:
adata = sc.read_10x_mtx(mtx_dir, var_names="gene_symbols", cache=False)
adata.var_names_make_unique()  # algunos símbolos de genes están duplicados
adata

### La anatomía de `AnnData`

```
                 var  (genes: gene_ids, highly_variable, ...)
              ┌───────────────────────────┐
  obs         │                           │   obsm: X_pca, X_umap (una fila por célula)
 (células:    │   X  (células × genes)    │   layers: counts, ... (misma forma que X)
  n_genes,    │                           │   uns: todo lo demás (colores, parámetros)
  cluster...) │                           │   obsp: grafos célula–célula (vecinos)
              └───────────────────────────┘
```
- `adata.X` → la matriz (aquí conteos crudos de UMIs, dispersa)
- `adata.obs` → un DataFrame de pandas con una fila **por célula** (métricas de calidad, clústeres, tipos celulares...)
- `adata.var` → un DataFrame de pandas con una fila **por gen**
- `adata.obsm`, `adata.layers`, `adata.uns`, `adata.obsp` → embeddings, matrices alternativas, metadatos, grafos

El equivalente en R es un objeto **Seurat** o **SingleCellExperiment**. Los comparamos en el notebook 05.

In [ ]:
print("cells x genes:", adata.shape)
print("first barcodes:", adata.obs_names[:3].tolist())
print("first genes:   ", adata.var_names[:3].tolist())
print("matrix type:   ", type(adata.X).__name__)
sparsity = 1 - adata.X.nnz / (adata.n_obs * adata.n_vars)
print(f"zeros in the matrix: {sparsity:.1%}")
adata.var.head()

### Números básicos por célula
El total de UMIs por célula (**tamaño de biblioteca**) y el número de genes detectados son lo primero que miramos.

In [ ]:
adata.obs["total_counts"] = np.asarray(adata.X.sum(axis=1)).ravel()
adata.obs["n_genes"] = np.asarray((adata.X > 0).sum(axis=1)).ravel()
adata.obs.describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(adata.obs["total_counts"], bins=60, color="#2a9d8f")
axes[0].set(xlabel="UMIs per cell", ylabel="cells")
axes[1].hist(adata.obs["n_genes"], bins=60, color="#e76f51")
axes[1].set(xlabel="genes per cell")
# Gráfico de rango ("rodilla"), como en web_summary.html
ranked = np.sort(adata.obs["total_counts"].values)[::-1]
axes[2].loglog(np.arange(1, len(ranked) + 1), ranked, color="#264653")
axes[2].set(xlabel="barcode rank", ylabel="UMIs")
plt.tight_layout()

> Aquí el gráfico de rodilla solo muestra las **células identificadas** (matriz filtrada). En `raw_feature_bc_matrix`
> también verías la larga cola de gotas vacías después de la rodilla.

### ¿Qué genes se llevan la mayoría de las lecturas?

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)

Dominan los genes mitocondriales (`MT-`), ribosomales (`RPS`/`RPL`) y genes muy abundantes como `MALAT1`.
Es normal, y por eso necesitamos control de calidad y normalización (siguiente notebook).

### 💡 Te toca a ti
1. ¿Cuántos UMIs tiene la **primera** célula? 2. ¿Qué gen tiene el mayor conteo **total** sumando todas las células?
*Pista:* `adata.X.sum(axis=0)` suma sobre las células.

In [ ]:
# Tu código aquí

<details><summary><b>Ver solución</b></summary>

```python
print(adata.obs["total_counts"].iloc[0])
gene_totals = np.asarray(adata.X.sum(axis=0)).ravel()
print(adata.var_names[gene_totals.argmax()])
```
</details>

## 4. 🔎 Extra: lo mismo en R (Seurat)
```r
library(Seurat)
counts <- Read10X("filtered_gene_bc_matrices/hg19/")   # o Read10X_h5("filtered_feature_bc_matrix.h5")
pbmc <- CreateSeuratObject(counts, project = "pbmc3k", min.cells = 3, min.features = 200)
pbmc
```
Para convertir entre ecosistemas: guarda `adata.write_h5ad("pbmc.h5ad")` y léelo en R con
`zellkonverter::readH5AD()` (SingleCellExperiment) o `schard::h5ad2seurat()`.

## 5. 🔎 Extra: analizar tus propios datos
Sube tu `filtered_feature_bc_matrix.h5` (ícono de carpeta a la izquierda → subir) o cópialo a Google Drive, y luego:
```python
adata = sc.read_10x_h5("filtered_feature_bc_matrix.h5")
adata.var_names_make_unique()
```
Para varias muestras, lee cada una, agrega `adata.obs["sample"] = "nombre"` y únelas con
`anndata.concat([...], label="sample")`.

---
## ✅ ¡Listo!
Ya sabes de dónde viene la matriz de conteos y cómo vive en Python. Siguiente: **01 · Control de calidad y preprocesamiento**.